# 🧠 GALILEO BRAIN V3 — Fine-tuning Qwen3-4B## ELAB Tutor — Andrea Marro — 2026### Qwen = cervello rapido che PILOTA, SENTE, RICORDA### Backend: DeepSeek R1 + Chat + Gemini → Qwen smista tutto**Istruzioni:**1. Esegui Cell 1 (installazione)2. **Riavvia il runtime** (Runtime → Riavvia)3. Esegui tutte le celle da 2 in poi4. Carica il dataset quando richiesto5. Attendi il training (~30-45 min su T4)6. Scarica il modello GGUF

In [ ]:
# Cell 1: Installazione Unsloth + dipendenzeimport sysif "google.colab" in sys.modules:    !pip install --no-deps trl peft accelerate bitsandbytes    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"    !pip install --no-deps xformers triton    !pip install transformers==4.56.2 trl==0.22.2 datasetsprint("✅ Installazione completata!")print("⚠️  ORA RIAVVIA IL RUNTIME: Runtime → Riavvia runtime")print("   Poi esegui dalla Cell 2 in poi")

In [ ]:
# Cell 2: Carica Modello + Configurazione LoRAfrom unsloth import FastLanguageModelimport torchmodel, tokenizer = FastLanguageModel.from_pretrained(    model_name="unsloth/Qwen3-4B-Instruct-2507",    max_seq_length=2048,    dtype=None,  # auto-detect    load_in_4bit=True,)model = FastLanguageModel.get_peft_model(    model,    r=64,                    # LoRA rank (alto per massima espressività)    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",                     "gate_proj", "up_proj", "down_proj"],    lora_alpha=128,          # 2:1 ratio per learning signal forte    lora_dropout=0.05,       # Regolarizzazione leggera    bias="none",    use_gradient_checkpointing="unsloth",    random_state=42,)print(f"✅ Modello caricato: Qwen3-4B-Instruct")print(f"   LoRA rank={64}, alpha={128}, dropout=0.05")print(f"   Parametri trainabili: {model.print_trainable_parameters()}")

In [ ]:
# Cell 3: Carica Datasetfrom google.colab import filesimport jsonprint("📂 Carica il file galileo-brain-v4-toolcall.jsonl")print("   (10,284 esempi con tool calling programmatico)")uploaded = files.upload()# Trova il file caricatodataset_file = list(uploaded.keys())[0]print(f"\n✅ File caricato: {dataset_file}")# Verifica il datasetwith open(dataset_file, 'r') as f:    lines = f.readlines()    print(f"   Totale esempi: {len(lines)}")# Mostra un esempiosample = json.loads(lines[0])print(f"\n📋 Esempio #1:")print(f"   User: {sample['messages'][1]['content'][:100]}...")print(f"   Assistant: {sample['messages'][2]['content'][:100]}...")

In [ ]:
# Cell 4: Prepara Dataset per Trainingfrom datasets import Datasetimport json# Carica tutti gli esempiexamples = []with open(dataset_file, 'r') as f:    for line in f:        examples.append(json.loads(line.strip()))dataset = Dataset.from_list(examples)# Split train/eval 90/10split = dataset.train_test_split(test_size=0.1, seed=42)train_dataset = split["train"]eval_dataset = split["test"]print(f"✅ Dataset preparato:")print(f"   Training: {len(train_dataset)} esempi")print(f"   Evaluation: {len(eval_dataset)} esempi")# Formatta con chat templatedef formatting_func(examples):    convos = examples["messages"]    texts = []    for convo in convos:        text = tokenizer.apply_chat_template(            convo, tokenize=False, add_generation_prompt=False        )        texts.append(text)    return {"text": texts}train_dataset = train_dataset.map(formatting_func, batched=True)eval_dataset = eval_dataset.map(formatting_func, batched=True)print(f"\n📋 Esempio formattato:")print(train_dataset[0]["text"][:300] + "...")

In [ ]:
# Cell 5: Configura SFT Trainerfrom trl import SFTTrainer, SFTConfigfrom unsloth import is_bfloat16_supportedtrainer = SFTTrainer(    model=model,    tokenizer=tokenizer,    train_dataset=train_dataset,    eval_dataset=eval_dataset,    dataset_text_field="text",    max_seq_length=2048,    dataset_num_proc=2,    args=SFTConfig(        per_device_train_batch_size=4,        gradient_accumulation_steps=4,   # Effective batch = 16        warmup_steps=30,        num_train_epochs=3,        learning_rate=2e-4,        fp16=not is_bfloat16_supported(),        bf16=is_bfloat16_supported(),        logging_steps=10,        optim="adamw_8bit",        weight_decay=0.01,        lr_scheduler_type="cosine",        seed=42,        output_dir="outputs",        eval_strategy="steps",        eval_steps=25,        save_strategy="steps",        save_steps=50,        save_total_limit=2,        load_best_model_at_end=True,        metric_for_best_model="eval_loss",    ),)# Train SOLO su risposte dell'assistente (non su system/user)from unsloth.chat_templates import train_on_responses_onlytrainer = train_on_responses_only(    trainer,    instruction_part="<|im_start|>user\n",    response_part="<|im_start|>assistant\n",)print("✅ Trainer configurato:")print("   Batch: 4 × 4 = 16 effective")print("   Epochs: 3")print("   LR: 2e-4 con cosine scheduler")print("   Loss SOLO su risposte assistant")

In [ ]:
# Cell 6: TRAINING 🚀import timeprint("🚀 Avvio training...")print("   Tempo stimato: 30-45 minuti su T4")print("   Target: loss finale < 0.1")print("=" * 50)start = time.time()stats = trainer.train()elapsed = time.time() - startprint(f"\n✅ Training completato in {elapsed/60:.1f} minuti!")print(f"   Loss finale: {stats.training_loss:.4f}")print(f"   Steps totali: {stats.global_step}")

In [ ]:
# Cell 7: Test Inferenza — Verifica tutti i 6+ intentFastLanguageModel.for_inference(model)test_cases = [    # ACTION    {"context": "tab: simulator\ncomponenti: [led1, resistor1]\nfili: 2",      "message": "Fai partire la simulazione!",     "expected_intent": "action"},    # CIRCUIT    {"context": "tab: simulator\ncomponenti: []\nfili: 0",     "message": "Metti un LED rosso vicino alla batteria",     "expected_intent": "circuit"},    # CODE    {"context": "tab: editor\nesperimento: v1-cap6-esp1",     "message": "Scrivi il codice per far lampeggiare il LED",     "expected_intent": "code"},    # TUTOR    {"context": "tab: simulator\nvolume_attivo: 1",     "message": "Cos'è la legge di Ohm?",     "expected_intent": "tutor"},    # NAVIGATION    {"context": "tab: simulator\nvolume_attivo: 1",     "message": "Apri il volume 2 capitolo condensatori",     "expected_intent": "navigation"},    # VOICE    {"context": "tab: simulator",     "message": "Attiva la voce, voglio che mi parli",     "expected_intent": "voice"},    # DRAWING    {"context": "tab: simulator\ncomponenti: [led1]",     "message": "Fammi disegnare sulla breadboard",     "expected_intent": "drawing"},    # HARD MULTI-STEP    {"context": "tab: simulator\ncomponenti: [led1:rosso, resistor1:220]\nfili: 2",     "message": "Togli il LED rosso, metti uno verde, collegalo al pin D5 con un resistore da 330 ohm e fai partire",     "expected_intent": "circuit"},]print("🧪 Test Inferenza — 8 casi")print("=" * 60)passed = 0for i, tc in enumerate(test_cases):    messages = [        {"role": "system", "content": "Sei il BRAIN di ELAB Tutor. Rispondi SOLO in JSON con: intent, entities, tool_calls, needs_llm, response, llm_hint."},        {"role": "user", "content": f"[CONTESTO]\n{tc['context']}\n\n[MESSAGGIO]\n{tc['message']}"}    ]        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")        outputs = model.generate(        input_ids=inputs,        max_new_tokens=512,        temperature=0.1,        top_p=0.95,        do_sample=True,    )        response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)        try:        parsed = json.loads(response.strip().split('\n')[0] if '\n' in response else response)        intent_ok = parsed.get("intent") == tc["expected_intent"]        status = "✅" if intent_ok else "❌"        if intent_ok: passed += 1        print(f"{status} Test {i+1}: '{tc['message'][:40]}...'")        print(f"   Expected: {tc['expected_intent']} | Got: {parsed.get('intent')}")        if parsed.get("tool_calls"):            print(f"   Tools: {[t['name'] for t in parsed['tool_calls'][:3]]}")    except:        print(f"❌ Test {i+1}: JSON parse error")        print(f"   Raw: {response[:100]}...")print(f"\n{'='*60}")print(f"Score: {passed}/{len(test_cases)} ({100*passed/len(test_cases):.0f}%)")print(f"Target: ≥ 90%")

In [ ]:
# Cell 8: Evaluation Suite Rapida (50 test)import random# Genera test automaticieval_tests = []with open(dataset_file, 'r') as f:    all_examples = [json.loads(line) for line in f]# Campiona 50 random dall'eval setrandom.seed(42)sample_indices = random.sample(range(len(all_examples)), min(50, len(all_examples)))correct_intent = 0correct_tools = 0total = len(sample_indices)print(f"🧪 Evaluation su {total} esempi random")print("=" * 50)for idx in sample_indices:    example = all_examples[idx]    expected = json.loads(example["messages"][2]["content"])        messages = example["messages"][:2]  # system + user only        inputs = tokenizer.apply_chat_template(        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"    ).to("cuda")        outputs = model.generate(        input_ids=inputs, max_new_tokens=512, temperature=0.1, do_sample=True    )        response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)        try:        parsed = json.loads(response.strip().split('\n')[0] if '\n' in response else response)        if parsed.get("intent") == expected["intent"]:            correct_intent += 1        if parsed.get("tool_calls") and expected.get("tool_calls"):            pred_tools = set(t["name"] for t in parsed["tool_calls"])            exp_tools = set(t["name"] for t in expected["tool_calls"])            if pred_tools == exp_tools:                correct_tools += 1    except:        passprint(f"\n📊 Risultati:")print(f"   Intent accuracy: {100*correct_intent/total:.1f}%")print(f"   Tool accuracy:   {100*correct_tools/total:.1f}%")print(f"\n   Target: intent ≥ 90%, tools ≥ 85%")

In [ ]:
# Cell 9: Salva Modello + Export GGUFprint("💾 Salvataggio LoRA adapter...")model.save_pretrained("galileo-brain-v3-lora")tokenizer.save_pretrained("galileo-brain-v3-lora")print("✅ LoRA adapter salvato")print("\n📦 Export GGUF q4_k_m per Ollama...")model.save_pretrained_gguf(    "galileo-brain-v3-gguf",    tokenizer,    quantization_method="q4_k_m")print("✅ GGUF esportato!")# Verifica fileimport osgguf_files = [f for f in os.listdir("galileo-brain-v3-gguf") if f.endswith('.gguf')]for f in gguf_files:    size_mb = os.path.getsize(f"galileo-brain-v3-gguf/{f}") / (1024*1024)    print(f"   {f}: {size_mb:.1f} MB")

In [ ]:
# Cell 10: Scarica il modellofrom google.colab import filesimport osgguf_dir = "galileo-brain-v3-gguf"gguf_files = [f for f in os.listdir(gguf_dir) if f.endswith('.gguf')]if gguf_files:    gguf_path = os.path.join(gguf_dir, gguf_files[0])    print(f"📥 Download: {gguf_files[0]}")    files.download(gguf_path)    print("✅ Download avviato!")    print("\n📋 Per usare con Ollama:")    print("   1. Copia il file .gguf nella cartella models/")    print("   2. Crea un Modelfile:")    print('      FROM ./galileo-brain-v3-gguf.Q4_K_M.gguf')    print("   3. ollama create galileo-brain -f Modelfile")    print("   4. ollama run galileo-brain")else:    print("❌ Nessun file GGUF trovato")